# 08 — Qualitative Error Analysis (Rebuttal Experiments)
Neste notebook comparamos as predições de M0 e M2 de todos os folds para extrair exemplos representativos sem viés de seleção (cherry-picking).

In [ ]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_recall_curve
import random

# Definindo seed para reprodutibilidade da amostragem
np.random.seed(42)
random.seed(42)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/workspace')
    IN_COLAB = True
except:
    ROOT = Path(r"/workspace")
    IN_COLAB = False

OUT_DIR     = ROOT / 'project'
SPLITS_DIR  = OUT_DIR / "splits"
MODELS_DIR  = OUT_DIR / "models"
CORE_COLS   = ["ENANTEMA", "PÓLIPO", "ÚLCERA", "EROSÃO", "MICRONODULARIDADE"]
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EndoDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.labels = self.df[CORE_COLS].values.astype(np.float32)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = str(ROOT / "Dev" / "Data" / "Imgs" / self.df.loc[idx, "image_name"])
        try: img = Image.open(img_path).convert("RGB")
        except: img = Image.new("RGB", (224, 224))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def get_dataloaders(fold):
    df_va = pd.read_csv(SPLITS_DIR / f"fold_{fold}_val.csv").fillna(0)
    df_te = pd.read_csv(SPLITS_DIR / f"fold_{fold}_test.csv").fillna(0)
    tf_te = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    ldr_va = DataLoader(EndoDataset(df_va, tf_te), batch_size=32, shuffle=False)
    ldr_te = DataLoader(EndoDataset(df_te, tf_te), batch_size=32, shuffle=False)
    return ldr_va, ldr_te

def inference(model, loader):
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.sigmoid(model(x.to(DEVICE)))
            all_p.append(p.cpu().numpy())
            all_t.append(y.numpy())
    return np.vstack(all_p), np.vstack(all_t)

def get_thresholds(p_va, t_va):
    thrs = []
    for c in range(p_va.shape[1]):
        if np.sum(t_va[:, c]) == 0:
            print(f"  [Aviso] Classe {CORE_COLS[c]} sem positivos na validação! Usando 0.5")
            thrs.append(0.5)
            continue
        pr, rc, th = precision_recall_curve(t_va[:,c], p_va[:,c])
        f1 = 2 * (pr * rc) / (pr + rc + 1e-8)
        best_th = th[np.argmax(f1)] if len(th) > 0 else 0.5
        thrs.append(best_th)
    return np.array(thrs)

def load_resnet():
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, len(CORE_COLS)))
    return m


In [ ]:
# ── 2. Inferência Coletiva (Todos os 5 Folds) ──────────────────────────────
print("Iniciando inferência em 5 folds...")

all_results = []

for fold in range(5):
    ldr_va, ldr_te = get_dataloaders(fold)
    
    # Carrega M0 (sem dict_key "model_state_dict" pois salvamos raw no notebook 03)
    m0 = load_resnet()
    m0.load_state_dict(torch.load(MODELS_DIR / f"M0_BCE_fold{fold}.pth", map_location=DEVICE, weights_only=True))
    m0.to(DEVICE)
    p0_va, t0_va = inference(m0, ldr_va)
    p0_te, t0_te = inference(m0, ldr_te)
    th0 = get_thresholds(p0_va, t0_va)
    y0_pred = (p0_te >= th0).astype(int)
    
    # Carrega M2 (tem dict_key "model_state_dict" pois salvamos assim no notebook 05)
    m2 = load_resnet()
    m2.load_state_dict(torch.load(MODELS_DIR / f"M2_fold{fold}.pt", map_location=DEVICE, weights_only=True)["model_state_dict"])
    m2.to(DEVICE)
    p2_va, t2_va = inference(m2, ldr_va)
    p2_te, t2_te = inference(m2, ldr_te)
    th2 = get_thresholds(p2_va, t2_va)
    y2_pred = (p2_te >= th2).astype(int)
    
    # Blindagem técnica: Garantir que a ordem dos arrays é idêntica
    assert np.array_equal(t0_te, t2_te), f"Mismatch ground truth fold {fold}"
    
    for i in range(len(t0_te)):
        all_results.append({
            'fold': fold, 'idx': i,
            'gt': t0_te[i], 'p0': y0_pred[i], 'p2': y2_pred[i]
        })

print(f"Total de amostras processadas: {len(all_results)}")


In [ ]:
# ── 3. Amostragem Sem Viés (Blind Sampling) ────────────────────────────────
def labels_to_str(binary_vec):
    active = [CORE_COLS[i].capitalize() for i in range(5) if binary_vec[i] == 1]
    return ", ".join(active) if active else "None"

pool_rara = []
pool_cooc = []
pool_fp = []
pool_sup = []

for r in all_results:
    gt, p0, p2 = r['gt'], r['p0'], r['p2']
    
    # 1. Rare class recovered (Polyp or Micronodularity)
    if (gt[1]==1 and p2[1]==1 and p0[1]==0) or (gt[4]==1 and p2[4]==1 and p0[4]==0):
        pool_rara.append((labels_to_str(gt), labels_to_str(p0), labels_to_str(p2), "Rare label recovered by M2; missed by M0."))
        
    # 2. Multilabel consistency (M2 gets full co-occurrence, M0 misses some)
    if sum(gt) > 1 and np.array_equal(p2, gt) and not np.array_equal(p0, gt):
        pool_cooc.append((labels_to_str(gt), labels_to_str(p0), labels_to_str(p2), "Full multilabel context captured by M2."))
        
    # 3. False positive associated with coupling (Erosion or Ulcer)
    if (gt[2]==0 and p2[2]==1 and p0[2]==0) or (gt[3]==0 and p2[3]==1 and p0[3]==0):
        pool_fp.append((labels_to_str(gt), labels_to_str(p0), labels_to_str(p2), "False positive by M2 (potentially coupled finding)."))
        
    # 4. Suppression of prevalent label
    if (gt[0]==1 and p2[0]==0 and p0[0]==1) or (gt[3]==1 and p2[3]==0 and p0[3]==1):
        pool_sup.append((labels_to_str(gt), labels_to_str(p0), labels_to_str(p2), "Prevalent finding suppressed by M2."))

# Seleção de 1 caso de cada (aleatório via seed fixa)
final_cases = []
for pool in [pool_rara, pool_cooc, pool_fp, pool_sup]:
    if len(pool) > 0:
        final_cases.append(random.choice(pool))

print(f"Total de candidatos encontrados:\nRaros: {len(pool_rara)}\nCo-ocorrência: {len(pool_cooc)}\nFalsos Pos: {len(pool_fp)}\nFalsos Neg: {len(pool_sup)}")


In [ ]:
# ── 4. Geração do Código LaTeX (UTF-8) ─────────────────────────────────────
latex_code = """
\\begin{table}[htbp]
\\centering
\\caption{Representative test cases comparing M0\\_BCE and M2\\_coo predictions across all folds.}
\\label{tab:error-analysis}
\\small
\\begin{tabular}{p{0.22\\linewidth}p{0.22\\linewidth}p{0.22\\linewidth}p{0.24\\linewidth}}
\\hline
\\textbf{Ground truth} & \\textbf{M0\\_BCE} & \\textbf{M2\\_coo} & \\textbf{Interpretation} \\\\
\\hline
"""

for c in final_cases:
    # Como \usepackage[utf8]{inputenc} é padrão moderno, 
    # mantemos os acentos e apenas formatamos a quebra de linha visual.
    gt = c[0]
    p0 = c[1]
    p2 = c[2]
    inter = c[3]
    
    latex_code += f"{gt} & {p0} & {p2} & {inter} \\\\\n"

latex_code += """\\hline
\\end{tabular}
\\end{table}
"""

print(latex_code)
